## Test the chain: Raw -> Enriched -> ArbFree -> OptionFilter


In [ ]:

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree
%aimport quantdata.widb.options_filter

from datetime import date
from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree
from quantdata.widb.options_filter import OptionsFilter, filter_options_day
from quantdata.biz_calendar import next_business_day
from cdxcore.verbose import Context
from IPython.display import display, Markdown


def test():
    ticker = "SPX"
    d      = next_business_day( date(2020, 1, 28), count_start=True )

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    verbose = Context.all

    display(Markdown("## collect_raw_day"))
    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, override_cache_mode="off" )
    display(raw_day)

    display(Markdown("## enrich_day"))
    enr_day = enrich_day(ticker, d, raw_day, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, override_cache_mode="off" )
    display(enr_day)    
    display(enr_day[['nms','nms_atm']])

    display(Markdown("## mk_arb_free_day"))
    mk_arb_free = MkArbFree()
    arb_free = mk_arb_free_day(ticker, d, enr_day, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, override_cache_mode="off")
    display(arb_free)

    display(Markdown("## filter_options_day"))
    options_filter = OptionsFilter()
    opt_flt = filter_options_day( ticker, d, arb_free, options_filter = options_filter, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose , override_cache_mode="off")
    display(opt_flt)
    return raw_day, enr_day, arb_free, opt_flt

raw_day, enr_day, arb_free, opt_flt = test()

In [ ]:
print( raw_day['tte_y'].unique() )
print( raw_day['expiry'].unique() )
print( enr_day['tte_y'].unique() )
print( arb_free['tte_y'].unique() )
print( arb_free['expiry'].unique() )
print( opt_flt['tte_y'].unique() )

In [ ]:
arb_free.columns